# Mage-Flow-Turbo-Native-Inference
### Portable Kaggle qualification demo (thin adapter around the generic core)

This notebook clones the portable native inference repository and invokes the **generic core** through `integrations/kaggle/qualification.py`. It does not re-implement inference here.

**BACKEND selection:** set `QUALIFICATION_BACKEND` to `cpu` (default, CPU session) or `cuda0` (NVIDIA GPU session). Accelerator=None for CPU; T4/T4x2 for CUDA.

**4 required inputs:**

| Component | Kaggle input / variation |
|---|---|
| DiT | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — GGUF / q8-0 |
| Qwen3-VL 4B | `dangkhoa2016/qwen-qwen3-vl-4b-instruct-gguf` — GGUF / q4-k-m |
| VAE | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — PyTorch / vae-only |
| CPU runtime (CPU only, optional) | `dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime` |

> Tiếng Việt: Notebook này clone repository native inference và gọi core chung qua `integrations/kaggle/qualification.py`. Đặt `QUALIFICATION_BACKEND=cpu` (CPU) hoặc `cuda0` (GPU).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-Native-Inference.git"
WORK = Path("/kaggle/working")
CHECKOUT = WORK / "Mage-Flow-Turbo-Native-Inference"
QUALIFICATION_BACKEND = os.environ.get("QUALIFICATION_BACKEND", "cpu")

if CHECKOUT.exists():
    subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--all"], check=True, capture_output=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CHECKOUT)], check=True, capture_output=True)

SOURCE_HEAD = subprocess.run(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f"SOURCE_HEAD={SOURCE_HEAD}")
print(f"QUALIFICATION_BACKEND={QUALIFICATION_BACKEND}")
sys.path.insert(0, str(CHECKOUT))

In [ ]:
from pathlib import Path
from integrations.kaggle.qualification import run_qualification

WORK_ROOT = WORK / "mageflow-qualification"
evidence = run_qualification(
    input_root=Path("/kaggle/input"),
    work_root=WORK_ROOT,
    backend=QUALIFICATION_BACKEND,
    repo_dir=CHECKOUT,
)
print("QUALIFICATION=PASS")
import json
print(json.dumps(evidence, indent=2))

## Results
The generated PNG and evidence JSON are stored under `/kaggle/working/mageflow-qualification/output/`.

Display the PNG and the qualification evidence path below.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

OUT = Path("/kaggle/working/mageflow-qualification/output")
png = OUT / f"qual-{QUALIFICATION_BACKEND}.png"
if png.exists():
    display(Image(filename=str(png)))
    print(f"PNG: {png}")
else:
    print("No canonical PNG found; check run_qualification output.")

from IPython.display import FileLink
ev = OUT / f"qualification-{QUALIFICATION_BACKEND}.json"
if ev.exists():
    display(FileLink(str(ev)))